# {ANALYSIS_TITLE}

**Analysis type:** {ANALYSIS_TYPE}  
**Target accession:** set `ACCESSION` in Section 2  
**Created:** {DATE}  

---

## Sections
1. Environment setup
2. Accession & configuration
3. AFDB API data fetch
4. Structure parsing
5. Metric computation
6. Visualisation
7. Flywheel result submission *(optional)*

**Dependencies:** `numpy`, `matplotlib`, `seaborn`, `requests`, `molviewspec`, `ipywidgets`  
**Prohibited:** `biopython`, `torch`, `torch-geometric`

---
## 1. Environment Setup

In [ ]:
# Install dependencies (Colab: run once; local venv: skip if already installed)
import subprocess, sys

PACKAGES = [
    'numpy',
    'matplotlib',
    'seaborn',
    'requests',
    'molviewspec',
    'ipywidgets',
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('Installation complete.')

In [ ]:
import base64
import io
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import requests
import seaborn as sns
from IPython.display import HTML, IFrame, display

sns.set_style('white')
warnings.filterwarnings('ignore')
print('Imports OK.')

---
## 2. Accession & Configuration

Set `ACCESSION` to an AlphaFold DB accession (e.g. `AF-0000000065889468`)  
or set `USE_LOCAL_FILE = True` and upload your own mmCIF/PAE/pLDDT files.

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
ACCESSION     = 'AF-0000000065889468'   # AFDB accession for a homodimer
USE_LOCAL_FILE = False                   # True → upload widgets appear below
DIST_CUTOFF    = 8.0                     # CB–CB contact cutoff in Å
# ──────────────────────────────────────────────────────────────────────────────

AFDB_META_URL = 'https://alphafold.ebi.ac.uk/api/prediction/{acc}'
print(f'Accession: {ACCESSION}')

In [ ]:
# Local file upload (only shown when USE_LOCAL_FILE = True)
if USE_LOCAL_FILE:
    import ipywidgets as widgets
    upload_cif    = widgets.FileUpload(description='mmCIF',   multiple=False)
    upload_pae    = widgets.FileUpload(description='PAE JSON', multiple=False)
    upload_plddt  = widgets.FileUpload(description='pLDDT JSON', multiple=False)
    display(upload_cif, upload_pae, upload_plddt)
    print('Upload files then run the next cell.')
else:
    print('Online mode: data will be fetched from AFDB REST API.')

---
## 3. AFDB API Data Fetch

Fetches metadata from `https://alphafold.ebi.ac.uk/api/prediction/{accession}`,
then downloads mmCIF, PAE JSON, and pLDDT JSON via the URLs in that response.

In [ ]:
def _get_json(url, timeout=30):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()

def _get_text(url, timeout=60):
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    return r.text

if USE_LOCAL_FILE:
    # Read from upload widgets (run after uploading in Section 2)
    cif_text  = list(upload_cif.value.values())[0]['content'].decode()
    pae_raw   = json.loads(list(upload_pae.value.values())[0]['content'].decode()) if upload_pae.value else None
    plddt_raw = json.loads(list(upload_plddt.value.values())[0]['content'].decode()) if upload_plddt.value else None
    meta      = {}
    bcif_url  = None
    print('Loaded from local files.')
else:
    meta_list = _get_json(AFDB_META_URL.format(acc=ACCESSION))
    meta      = meta_list[0] if isinstance(meta_list, list) else meta_list

    cif_url   = meta['cifUrl']
    bcif_url  = meta.get('bcifUrl')         # used only by MolViewSpec viewer
    pae_url   = meta['paeDocUrl']
    plddt_url = meta['plddtDocUrl']

    print('Downloading mmCIF …')
    cif_text  = _get_text(cif_url)
    print('Downloading PAE JSON …')
    pae_raw   = _get_json(pae_url)
    print('Downloading pLDDT JSON …')
    plddt_raw = _get_json(plddt_url)
    print('All downloads complete.')

# --- Display metadata ---
if meta:
    print(f"\nProtein : {meta.get('proteinFullName') or meta.get('uniprotDescription', 'N/A')}")
    print(f"Organism: {meta.get('organismScientificName') or meta.get('organism', 'N/A')}")
    print(f"Gene    : {meta.get('geneNames', 'N/A')}")
    print(f"Model   : {meta.get('modelVersion', 'N/A')}")
    print(f"Length  : {len(meta.get('sequence', '')) or 'N/A')} aa (monomer)")

---
## 4. Structure Parsing

Parses mmCIF into per-chain CB/CA coordinate arrays and extracts PAE + pLDDT arrays.

In [ ]:
# ── mmCIF PARSER ──────────────────────────────────────────────────────────────

def parse_mmcif_atoms(cif_text):
    """Locate the _atom_site loop and return (records, col_idx)."""
    lines = cif_text.splitlines()
    loop_start = None
    for i, line in enumerate(lines):
        if line.strip().lower() == 'loop_':
            j = i + 1
            hdrs = []
            while j < len(lines) and lines[j].strip().startswith('_'):
                hdrs.append(lines[j].strip())
                j += 1
            if any(h.lower().startswith('_atom_site.') for h in hdrs):
                loop_start = i
                break
    if loop_start is None:
        return [], {}
    j = loop_start + 1
    hdrs = []
    while j < len(lines) and lines[j].strip().startswith('_'):
        hdrs.append(lines[j].strip().lower())
        j += 1
    col_idx = {h: idx for idx, h in enumerate(hdrs)}
    records = []
    while j < len(lines):
        s = lines[j].strip()
        if not s or s.startswith('#') or s.startswith('_') or s.startswith('loop_'):
            break
        records.append(s.split())
        j += 1
    return records, col_idx


def extract_cb_coords(records, col_idx):
    """Return {chain_id: (coords_Nx3, res_ids, res_names, plddt)} dicts."""
    def c(name):
        return col_idx.get(f'_atom_site.{name}')

    ix_grp   = c('group_pdb')
    ix_atom  = c('label_atom_id')
    ix_chain = c('label_asym_id')
    ix_seq   = c('label_seq_id')
    ix_resn  = c('label_comp_id')
    ix_x, ix_y, ix_z = c('cartn_x'), c('cartn_y'), c('cartn_z')
    ix_b     = c('b_iso_or_equiv')
    ix_model = c('pdbx_pdb_model_num')

    per_chain = {}
    for toks in records:
        if toks[ix_grp] != 'ATOM':
            continue
        if ix_model is not None and toks[ix_model] != '1':
            continue
        atom = toks[ix_atom]
        if atom not in ('CA', 'CB'):
            continue
        res_str = toks[ix_seq]
        if res_str in ('.', '?') or not res_str.lstrip('-').isdigit():
            continue
        ch = toks[ix_chain]
        rid = int(res_str)
        try:
            xyz   = (float(toks[ix_x]), float(toks[ix_y]), float(toks[ix_z]))
            b     = float(toks[ix_b])
        except ValueError:
            continue
        per_chain.setdefault(ch, {})
        existing = per_chain[ch].get(rid)
        if existing is None or (atom == 'CB' and existing[0] == 'CA'):
            per_chain[ch][rid] = (atom, toks[ix_resn], *xyz, b)

    ch_coords, ch_resids, ch_resnames, ch_plddt = {}, {}, {}, {}
    for ch, res_dict in per_chain.items():
        sids = sorted(res_dict)
        ch_coords[ch]   = np.array([[res_dict[r][2], res_dict[r][3], res_dict[r][4]] for r in sids], dtype=np.float32)
        ch_resids[ch]   = np.array(sids, dtype=np.int32)
        ch_resnames[ch] = np.array([res_dict[r][1] for r in sids])
        ch_plddt[ch]    = np.array([res_dict[r][5] for r in sids], dtype=np.float32)
    return ch_coords, ch_resids, ch_resnames, ch_plddt


records, col_idx = parse_mmcif_atoms(cif_text)
ch_coords, ch_resids, ch_resnames, ch_plddt = extract_cb_coords(records, col_idx)
chain_ids = sorted(ch_coords.keys())
print(f'Chains found: {chain_ids}')
for ch in chain_ids:
    print(f'  Chain {ch}: {len(ch_coords[ch])} residues')

In [ ]:
# ── PAE + pLDDT ARRAYS ────────────────────────────────────────────────────────

# Chain lengths from PAE JSON (fallback to structure-derived)
if pae_raw and 'chains' in pae_raw[0]:
    pae_chains = sorted(pae_raw[0]['chains'], key=lambda c: c['label_asym_id'])
    nA = pae_chains[0]['sequenceEnd'] - pae_chains[0]['sequenceStart'] + 1
    nB = pae_chains[1]['sequenceEnd'] - pae_chains[1]['sequenceStart'] + 1
else:
    nA = len(ch_resids[chain_ids[0]])
    nB = len(ch_resids[chain_ids[1]])

pae_matrix = np.array(pae_raw[0]['predicted_aligned_error'], dtype=np.float32) if pae_raw else None
pae_max    = (pae_raw[0].get('max_predicted_aligned_error') or 31.75) if pae_raw else 31.75

if pae_matrix is not None:
    pae_AA = pae_matrix[:nA,      :nA]
    pae_AB = pae_matrix[:nA,      nA:nA+nB]
    pae_BA = pae_matrix[nA:nA+nB, :nA]
    pae_BB = pae_matrix[nA:nA+nB, nA:nA+nB]
    print(f'PAE matrix: {pae_matrix.shape}  nA={nA}  nB={nB}')

if plddt_raw:
    plddt_all = np.array(plddt_raw['confidenceScore'], dtype=np.float32)
    plddt_A   = plddt_all[:nA]
    plddt_B   = plddt_all[nA:nA+nB]
    print(f'pLDDT loaded: mean_A={plddt_A.mean():.1f}  mean_B={plddt_B.mean():.1f}')

---
## 5. Metric Computation

All scoring functions are inlined as pure NumPy. Each `compute_*` returns a flat dict
containing the final score **and** all intermediate values (masks, counts) to avoid
recomputation during visualisation.

In [ ]:
# ── PRIMITIVE FUNCTIONS ───────────────────────────────────────────────────────

def d0_func(L):
    if L <= 27:
        return 1.0
    return max(1.0, 1.24 * (L - 15) ** (1.0 / 3.0) - 1.8)

def ptm_func(pae_vals, d0):
    return 1.0 / (1.0 + (pae_vals / d0) ** 2)

In [ ]:
# ── INTERFACE DETECTION ───────────────────────────────────────────────────────
# Uses CB–CB distance (CA fallback for GLY, already handled by extract_cb_coords)

coords_A = ch_coords[chain_ids[0]]
coords_B = ch_coords[chain_ids[1]]

diff        = coords_A[:, np.newaxis, :] - coords_B[np.newaxis, :, :]
dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))   # (nA, nB)
contact_mat = dist_matrix <= DIST_CUTOFF
if_A        = contact_mat.any(axis=1)
if_B        = contact_mat.any(axis=0)
n_contacts  = int(if_A.sum()) + int(if_B.sum())

print(f'Interface contacts (CB–CB ≤ {DIST_CUTOFF} Å): {n_contacts} residues  '
      f'(A: {int(if_A.sum())}, B: {int(if_B.sum())})')

In [ ]:
# ── SCORE COMPUTATION ─────────────────────────────────────────────────────────
# Replace / extend this cell for non-homodimer analysis types

scores = {}

if pae_matrix is not None:
    # ipTM
    d0_iptm = d0_func(nA + nB)
    iptm = max(
        ptm_func(pae_AB, d0_iptm).mean(axis=1).max(),
        ptm_func(pae_BA, d0_iptm).mean(axis=1).max(),
    )
    scores['ipTM'] = float(iptm)

    # ipSAE (three d0 variants)
    PAE_CUT = 10.0
    n_dom_A = int((pae_AB < PAE_CUT).any(axis=1).sum())
    n_dom_B = int((pae_BA < PAE_CUT).any(axis=1).sum())
    n_dom   = max(n_dom_A + n_dom_B, 1)

    def _ipsae_half(block, n_self, n_other, mode):
        out = []
        for row in block:
            mask = row < PAE_CUT
            n0 = int(mask.sum())
            if n0 == 0:
                out.append(0.0)
                continue
            d0 = d0_func(n0 if mode == 'res' else
                          (n_self + n_other) if mode == 'chn' else n_dom)
            out.append(float(ptm_func(row[mask], d0).mean()))
        return np.array(out)

    for mode in ('res', 'chn', 'dom'):
        sA = _ipsae_half(pae_AB, nA, nB, mode)
        sB = _ipsae_half(pae_BA, nB, nA, mode)
        scores[f'ipSAE_d0{mode}'] = float(max(sA.max(), sB.max()))

    # LIS
    LIS_CUT = 12.0
    def _lis(block):
        m = block < LIS_CUT
        return float(((LIS_CUT - block[m]) / LIS_CUT).mean()) if m.any() else 0.0
    scores['LIS'] = max(_lis(pae_AB), _lis(pae_BA))

# pDockQ
if plddt_raw and n_contacts > 0:
    mean_plddt = np.concatenate([plddt_A[if_A], plddt_B[if_B]]).mean()
    x_pdockq   = mean_plddt * np.log10(max(n_contacts, 1))
    scores['pDockQ'] = float(0.724 / (1 + np.exp(-0.052 * (x_pdockq - 152.611))) + 0.018)
else:
    scores['pDockQ'] = 0.018

# pDockQ2
if pae_matrix is not None and plddt_raw and contact_mat.any():
    D0_FIXED  = 10.0
    pae_cont  = pae_AB[contact_mat]
    mean_ptm2 = float(ptm_func(pae_cont, D0_FIXED).mean())
    mean_pl2  = float(np.concatenate([plddt_A[if_A], plddt_B[if_B]]).mean())
    x_pdq2    = mean_pl2 * mean_ptm2
    scores['pDockQ2'] = float(0.715 / (1 + np.exp(-12.3 * (x_pdq2 - 0.605))) + 0.005)
else:
    scores['pDockQ2'] = 0.005

print('\nConfidence scores:')
for k, v in scores.items():
    print(f'  {k:18s} {v:.4f}')

In [ ]:
# ── TRAFFIC-LIGHT CLASSIFICATION ──────────────────────────────────────────────

THRESHOLDS = {
    'ipSAE_d0res': (0.6, 0.4),
    'ipTM':        (0.7, 0.5),
    'pDockQ':      (0.23, 0.09),
    'pDockQ2':     (0.5, 0.23),
    'LIS':         (0.15, 0.09),
}

def traffic_light(val, score_name):
    g, a = THRESHOLDS[score_name]
    if val >= g: return 'green',  'HIGH'
    if val >= a: return 'amber',  'MODERATE'
    return        'red',   'LOW'

COLOUR_MAP = {'green': '\033[92m', 'amber': '\033[93m', 'red': '\033[91m', 'reset': '\033[0m'}
print('\nClassification:')
for name in THRESHOLDS:
    if name in scores:
        col, label = traffic_light(scores[name], name)
        print(f'  {COLOUR_MAP[col]}{name:18s} {scores[name]:.4f}  [{label}]{COLOUR_MAP["reset"]}')

---
## 6. Visualisation

In [ ]:
# ── 2D: PAE HEATMAP ───────────────────────────────────────────────────────────

if pae_matrix is not None:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(pae_matrix, cmap='Greens_r', vmin=0, vmax=pae_max, origin='upper')
    ax.axhline(nA - 0.5, color='white', linewidth=1.2)
    ax.axvline(nA - 0.5, color='white', linewidth=1.2)
    ax.set_xlabel('Residue index')
    ax.set_ylabel('Residue index')
    ax.set_title(f'Predicted Aligned Error — {ACCESSION}')
    plt.colorbar(im, ax=ax, label='PAE (Å)')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 2D: pLDDT PROFILE ─────────────────────────────────────────────────────────

if plddt_raw:
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(range(1, nA + 1), plddt_A, color='#009688', label=f'Chain A (n={nA})')
    ax.plot(range(nA + 1, nA + nB + 1), plddt_B, color='#E91E63', label=f'Chain B (n={nB})')
    # Shade interface residues
    for i, is_if in enumerate(if_A):
        if is_if:
            ax.axvspan(i + 0.5, i + 1.5, color='#009688', alpha=0.15)
    for j, is_if in enumerate(if_B):
        if is_if:
            ax.axvspan(nA + j + 0.5, nA + j + 1.5, color='#E91E63', alpha=0.15)
    ax.axhline(70, color='gray', linestyle='--', linewidth=0.8, label='pLDDT=70')
    ax.set_xlabel('Residue index')
    ax.set_ylabel('pLDDT')
    ax.set_title('Per-residue pLDDT (shaded = interface)')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 3D: MOLVIEWSPEC VIEWER ────────────────────────────────────────────────────
# Wrapped in try/except — degrades gracefully if molviewspec is not installed.

try:
    import molviewspec as mvs

    COLOUR_A = '#009688'
    COLOUR_B = '#E91E63'
    COLOUR_IF = '#FFD600'

    def _build_struct(url, fmt):
        builder = mvs.create_builder()
        return builder, (
            builder
            .download(url=url)
            .parse(format=fmt)
            .model_structure()
        )

    def show_mol_view(state, label, width=950, height=550):
        encoded = base64.b64encode(state.molstar_html().encode()).decode()
        display(HTML(f'<h4 style="font-family:sans-serif">{label}</h4>'))
        display(IFrame(src=f'data:text/html;base64,{encoded}', width=width, height=height))

    if bcif_url:
        # View 1: cartoon coloured by chain
        builder, structure = _build_struct(bcif_url, 'bcif')
        for chain_label, colour in [(chain_ids[0], COLOUR_A), (chain_ids[1], COLOUR_B)]:
            (structure
             .component(selector=mvs.ComponentExpression(label_asym_id=chain_label))
             .representation(type='cartoon')
             .color(color=colour))
        show_mol_view(builder.get_state(), 'Chain colouring (A = teal, B = pink)')

        # View 2: interface highlighted
        builder2, structure2 = _build_struct(bcif_url, 'bcif')
        for chain_label, colour in [(chain_ids[0], COLOUR_A), (chain_ids[1], COLOUR_B)]:
            (structure2
             .component(selector=mvs.ComponentExpression(label_asym_id=chain_label))
             .representation(type='cartoon')
             .color(color=colour))
        # Colour interface residues yellow
        res_ids_A = ch_resids[chain_ids[0]]
        res_ids_B = ch_resids[chain_ids[1]]
        for idx, is_if in enumerate(if_A):
            if is_if:
                rid = int(res_ids_A[idx])
                (structure2
                 .component(selector=mvs.ComponentExpression(
                     label_asym_id=chain_ids[0], beg_label_seq_id=rid, end_label_seq_id=rid))
                 .representation(type='cartoon')
                 .color(color=COLOUR_IF))
        for idx, is_if in enumerate(if_B):
            if is_if:
                rid = int(res_ids_B[idx])
                (structure2
                 .component(selector=mvs.ComponentExpression(
                     label_asym_id=chain_ids[1], beg_label_seq_id=rid, end_label_seq_id=rid))
                 .representation(type='cartoon')
                 .color(color=COLOUR_IF))
        show_mol_view(builder2.get_state(), 'Interface residues highlighted (yellow)')
    else:
        print('No bcifUrl available — skipping 3D viewer.')

except ImportError:
    print('molviewspec not installed — skipping 3D views.')

---
## 7. Flywheel Result Submission *(optional)*

Uncomment and configure this cell to write results to a downstream sink
(database, CSV file, or API endpoint). Remove if not applicable to this
analysis type.

In [ ]:
# ── FLYWHEEL RESULT SUBMISSION ────────────────────────────────────────────────
# Configure FLYWHEEL_ENDPOINT and FLYWHEEL_API_KEY as environment variables
# or Colab Secrets before enabling.

FLYWHEEL_ENABLED = False   # ← set True to activate

if FLYWHEEL_ENABLED:
    import os

    result_payload = {
        'accession':  ACCESSION,
        'analysis':   '{ANALYSIS_TYPE}',
        'scores':     scores,
        'n_contacts': n_contacts,
        'nA':         nA,
        'nB':         nB,
    }

    endpoint = os.environ.get('FLYWHEEL_ENDPOINT', '')
    api_key  = os.environ.get('FLYWHEEL_API_KEY', '')

    if not endpoint:
        print('FLYWHEEL_ENDPOINT not set — skipping submission.')
    else:
        resp = requests.post(
            endpoint,
            json=result_payload,
            headers={'Authorization': f'Bearer {api_key}'},
            timeout=15,
        )
        resp.raise_for_status()
        print(f'Result submitted: HTTP {resp.status_code}')
else:
    print('Flywheel submission disabled. Set FLYWHEEL_ENABLED = True to activate.')